# 10 — Sensibilidade do alvo ao status do pedido

**Pergunta:** vale a pena trocar o alvo de *pedidos com pagamento aprovado, incluindo cancelados* (atual) para *só pedidos entregues*?  
**Lê:** `order_seller_revenue` e `corte` (01), `coorte` (03). **Não ajusta modelos e não altera nenhum artefato.**  
**Grava:** `sensibilidade_status_*.csv` e `sensibilidade_status_semanal.png`.

O status usado é o **final**, como estava na extração da base. Na data da previsão ele ainda não é conhecido para pedidos recentes: veja a seção de leitura no fim.

In [ ]:
# === SETUP (idêntico em todos os notebooks; só muda NOTEBOOK) ===
NOTEBOOK = "10_sensibilidade_status"
RAIZ = "/content/drive/MyDrive/tcc_fluxo"      # Colab. Localmente: caminho da pasta do projeto (ou deixe None)

import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip install -q -r {RAIZ}/requirements.txt
except ImportError:                              # fora do Colab
    RAIZ = os.environ.get("TCC_RAIZ") or None
    if RAIZ is None:
        from pathlib import Path
        RAIZ = str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").is_file()))
sys.path.insert(0, f"{RAIZ}/src")
%load_ext autoreload
%autoreload 2

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tcc_fluxo import iniciar
C, A = iniciar(RAIZ, notebook=NOTEBOOK)

In [ ]:
from tcc_fluxo import alvo, graficos

corte = A.carregar_json("corte")
TEST_END = pd.Timestamp(corte["TEST_END"])
osr = A.carregar_df("order_seller_revenue")
cohort_ids = A.carregar_df("coorte")["seller_id"]
R = alvo.comparar_status(osr, cohort_ids, C.PANEL_START, TEST_END, C.TRAIN_TARGET_END)
print(pd.Series(R["resumo"]).round(4))

### Composição do valor por status e perda por período

In [ ]:
display(R["por_status"]); display(R["por_periodo"])
A.tabela(R["por_status"], "sensibilidade_status_por_status.csv")
A.tabela(R["por_periodo"], "sensibilidade_status_por_periodo.csv")

### Semana a semana e vendedor a vendedor

In [ ]:
ps = R["por_semana"]
A.tabela(ps, "sensibilidade_status_por_semana.csv")
A.tabela(R["por_vendedor"], "sensibilidade_status_por_vendedor.csv")
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(ps["week_start"], ps["pct_removido"], color=graficos.C_CHOSEN, lw=2)
ax.axvline(C.TRAIN_TARGET_END, color=graficos.INK2, ls="--", lw=1)
ax.text(C.TRAIN_TARGET_END, ax.get_ylim()[1] * 0.95, " início do teste", color=graficos.INK2, va="top", fontsize=9)
ax.set_ylabel("% do valor removido")
ax.set_title("Valor da coorte que sai do alvo se ficarem só pedidos entregues, por semana de aprovação", loc="left")
graficos.clean(ax); fig.tight_layout()
A.figura(fig, "sensibilidade_status_semanal.png"); plt.show()
print("Vendedores mais afetados:"); display(R["por_vendedor"].head(10))

### Como ler
- **Se o valor removido for pequeno e estável** (total semanal quase igual, correlação perto de 1), a troca não muda as conclusões no agregado; manter o alvo atual e citar o percentual como limitação é suficiente.
- **Se a perda se concentrar em poucos vendedores**, a troca muda a série justamente deles: vale reportar quantos.
- **Defasagem de informação:** o status "delivered" só é conhecido dias ou semanas depois da aprovação. Para usar *só entregues* sem vazamento, os atributos da semana *t* teriam de usar o status conhecido em *t*, não o final. Trocar o alvo exige refazer o 01 com essa regra e rodar tudo de novo numa nova `EXECUCAO`; os números não seriam comparáveis um a um com os atuais, porque o alvo muda.

In [ ]:
A.finalizar()